## 6.5 池化层（Pooling Layer） - 全局池化（Global Pooling）

#### 1. 什么是全局池化

##### 1.1 基本定义
前面我们学习的最大池化和平均池化，通常都是在一个局部小窗口中进行的，例如：
* 2 × 2
* 3 × 3

而全局池化（Global Pooling）则不同，它不是看局部小块，而是：

>直接对一整个特征图做一次池化。

也就是说，对每一个通道来说，它会把这一整张特征图整体压缩成 1 个值。

##### 1.2 一个最直观的理解
假设某一层输出的特征图形状是：

`7 × 7 × 16`

这表示：
* 高度 = 7
* 宽度 = 7
* 通道数 = 16

如果使用全局池化，那么对于每一个通道：
* 不再分很多小窗口
* 而是直接对这一整个 7 × 7 做一次池化

最终输出就会变成：

`1 × 1 × 16`

也可以简单理解成：

每个通道最后只留下 1 个代表值。

##### 1.3 它和普通池化的核心区别
普通池化：
* 看局部区域
* 每个区域输出一个值
* 最后得到一张更小的特征图

全局池化：
* 直接看整个通道
* 每个通道只输出一个值
* 最后得到的是“每个通道一个总结值”

所以你可以先记成：
* 普通池化：局部压缩
* 全局池化：整张总结

#### 2. 为什么需要全局池化

##### 2.1 进一步压缩特征图
全局池化的最直接作用就是：

>把每个通道的空间维度彻底压缩掉。

例如：
* 输入：7 × 7 × 64
* 全局池化后：1 × 1 × 64

这意味着：
* 高和宽都被压成了 1
* 只保留通道维度的信息

这样可以大大减少后续计算量。

##### 2.2 替代部分全连接层
在早期 CNN 中，卷积层后面常常会接：
* 展平（Flatten）
* 多个全连接层（Fully Connected）

但这样做有一个明显问题：

参数量可能会很大。

而全局池化可以直接把空间维度压缩掉，

让后面不需要特别大的全连接层，

从而减少参数量，降低过拟合风险。

##### 2.3 让模型更关注“有没有这个特征”
全局池化有一个很重要的思想：

>它不再特别关心这个特征具体出现在什么位置，而更关心这个通道整体上有没有强响应。

也就是说：
* 某个通道如果负责检测“竖线”
* 那么全局池化之后，我们更关心这一整张特征图里“竖线特征整体强不强”

所以它会让网络更偏向关注：

>特征是否存在、整体有多强

>而不是特别在意精确位置。

#### 3. 全局池化的两种常见方式

##### 3.1 全局最大池化（Global Max Pooling）
全局最大池化的规则是：

>对整张特征图取最大值。

也就是说，一个通道中所有位置里，只保留最大的那个响应值。

##### 3.2 全局平均池化（Global Average Pooling）
全局平均池化的规则是：

>对整张特征图取平均值。

也就是说，一个通道中所有位置的值都参与计算，最后得到一个平均值

##### 3.3 最常见的是谁

在现代 CNN 中，更常见、更重要的是：

>Global Average Pooling（全局平均池化，简称 GAP）

#### 4. 全局平均池化（Global Average Pooling）

##### 4.1 基本思想
对于某一个通道的整张特征图，

把其中所有数值加起来，再除以总元素个数，得到一个平均值。

例如某个通道的特征图是：
```
2  4
6  8
```

那么全局平均池化输出就是：

`(2 + 4 + 6 + 8) / 4 = 5`

##### 4.2 它在表达什么
它表达的是：

>这个通道整体上的平均响应强度。

也就是说，它不会只看最强的那个点，

而是看这一整张特征图整体上“有多活跃”。

##### 4.3  为什么它很常见
因为它有几个很大的优点：
* 参数少
* 结构简单
* 不容易过拟合
* 很适合在卷积层最后做整体汇总

所以在很多现代网络中，

最后都会把卷积输出接一个 GAP，再接分类层。

#### 5. 全局最大池化（Global Max Pooling）

##### 5.1 基本思想
对于某一个通道的整张特征图，

直接取其中最大的那个值。

例如某个通道是：
```
2  4
6  8
```
那么全局最大池化输出就是：

`8`

##### 5.2 它在表达什么
它表达的是：

这个通道里最强的一次响应是多少。

所以它更强调：

>只要这个特征在某个位置特别强地出现过，就把它保留下来。

##### 5.3 它和 GAP 的风格区别
* Global Max Pooling：关注最强一次出现 ⭐
* Global Average Pooling：关注整体平均水平 📘

#### 6. 一个完整的形状变化例子

##### 6.1 输入形状
假设某一层卷积输出为：

`8 × 8 × 32`
表示：
* 32 个通道
* 每个通道是一张 8 × 8 的特征图

##### 6.2 经过全局平均池化
如果使用 Global Average Pooling，那么：
* 每个 8 × 8 通道 → 压缩成 1 × 1
* 一共 32 个通道

所以输出变成：

`1 × 1 × 32`

很多时候也会写成：

`32`

也就是长度为 32 的特征向量。

##### 6.3 经过全局最大池化
如果使用 Global Max Pooling，结果的形状也是：

`1 × 1 × 32`

区别不在于形状，而在于：
* GAP 取平均
* GMP 取最大

#### 7. 结合 CNN 分类任务来理解

##### 7.1 最后一层卷积输出
假设一个 CNN 在最后一层卷积后，输出：

`7 × 7 × 64`

这表示模型已经提取出了 64 类高级特征。

例如你可以粗略理解为：
* 某些通道关注边缘组合
* 某些通道关注局部结构
* 某些通道关注更高层语义模式

##### 7.2 如果直接展平
如果直接 Flatten，那么会变成：

`7 × 7 × 64 = 3136`

也就是说后面全连接层要接收 3136 个输入，

参数量就会明显上升。

##### 7.3 如果改用全局平均池化
如果先做 GAP，那么直接变成：

`64`

这样后面只需要一个较小的线性层，

例如：
* 64 → 10（10 类分类）

##### 7.4 实际意义
所以全局池化常常出现在网络靠后的位置，

用来做：

卷积特征的最终汇总

它的逻辑是：
* 前面卷积层负责提取越来越高级的特征
* 最后全局池化负责把每个通道的整体信息总结成一个值
* 再交给分类层输出结果

#### 8. 全局池化的优点
**1️⃣ 参数更少**

因为它通常能减少甚至替代大规模全连接层，

所以模型参数量会明显减少。

---

**2️⃣ 更不容易过拟合**

参数少通常意味着模型更轻，

也更不容易在训练集上记得太死。

---

**3️⃣ 结构更简洁**

相比`卷积输出 → Flatten → 很大全连接层`，

`卷积输出 → Global Pooling → 分类层`通常更自然、更简洁。

---

**4️⃣ 更强调特征存在性**

它更关注某个特征在整个通道上的整体表现，

这对很多分类任务是有帮助的。

#### 9. 全局池化的局限
**1️⃣ 会丢失空间位置信息**

因为它直接把整张特征图压成 1 个值，

所以很多精细的空间位置信息会被丢掉。

这意味着它不太适合那些特别依赖精确位置信息的任务。

---

**2️⃣ 更适合靠后层使用**

正因为它压缩得太彻底，

所以一般不会在网络很前面就使用全局池化。

它更常见于：

网络后部、接近输出层的位置